In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('../../Mech_Interp/llm-transparency-tool')

In [2]:
# Typeguard monkey patch

import typeguard

def no_op_typechecked(target=None, **kwargs):
    if target is not None and callable(target):
        return target
    def wrapper(func):
        return func
    return wrapper

typeguard.typechecked = no_op_typechecked

In [3]:
import torch
import numpy as np
import collections
from typing import Dict, List, Tuple
import json
from tqdm import tqdm
import warnings
import os
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel, BitsAndBytesConfig, AutoModelForCausalLM
from notebooks.utils import create_prompts_nq
import pandas as pd

import llm_transparency_tool.models.tlens_model as tlens_model_module
import llm_transparency_tool.routes.contributions as contributions
from llm_transparency_tool.models.tlens_model import TransformerLensTransparentLlm

warnings.filterwarnings("ignore", category=UserWarning)

device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [4]:
# TransformerLens monkey patch

from transformer_lens import HookedTransformer

if not hasattr(HookedTransformer, "_original_set_use_attn_in"):
    HookedTransformer._original_set_use_attn_in = HookedTransformer.set_use_attn_in

    def safe_set_use_attn_in(self, use_attn_in):
        # If model uses GQA (n_key_value_heads is set) and we are just trying to disable attn_in
        if self.cfg.n_key_value_heads is not None and not use_attn_in:
            return # Silently skip to avoid Assertion Error
        # Otherwise run original logic
        return self._original_set_use_attn_in(use_attn_in)

    HookedTransformer.set_use_attn_in = safe_set_use_attn_in
    
# Streamlit monkey patch

# def custom_load_hooked_transformer(model_name, hf_model=None, tlens_device=None, **kwargs):
#     from transformer_lens import HookedTransformer
#     print(f"Custom loader: Initializing HookedTransformer for {model_name} with 4-bit compatibility flags...")
#     return HookedTransformer.from_pretrained(
#         model_name,
#         hf_model=hf_model,
#         device=tlens_device,
#         # CRITICAL: Disable all weight processing for 4-bit models
#         fold_ln=False,
#         fold_value_biases=False,
#         center_writing_weights=False,
#         center_unembed=False,
#         **kwargs
#     )

# # Apply the patch
# if hasattr(tlens_model_module, 'load_hooked_transformer'):
#     tlens_model_module.load_hooked_transformer = custom_load_hooked_transformer

if hasattr(tlens_model_module, 'load_hooked_transformer'):
    if hasattr(tlens_model_module.load_hooked_transformer, "__wrapped__"):
        tlens_model_module.load_hooked_transformer = tlens_model_module.load_hooked_transformer.__wrapped__

In [5]:
MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct" 
N_LAYERS = 32 
N_HEADS = 32
CONTRIBUTION_THRESHOLD = 0.03
DATASET_ID = "florin-hf/nq_open_gold" 

In [6]:
def get_target_token_id(tokenizer, answer: str) -> int:
    """Tokenizes the answer and returns the ID of the first token."""
    encoded_answer = tokenizer.encode(answer, add_special_tokens=False)
    if not encoded_answer:
        raise ValueError("Could not tokenize the target answer.")
    return encoded_answer[0]


def construct_prompts(question: str, context: str) -> Tuple[str, str]:
    """Creates the RAG and Non-RAG prompt templates."""
    # RAG (Open-Book) Prompt
    rag_prompt = (
        f"Context: {context}\n\n"
        f"Question: {question}\n\n"
        f"Answer:"
    )
    # Non-RAG (Closed-Book) Prompt
    non_rag_prompt = (
        f"Question: {question}\n\n"
        f"Answer:"
    )
    return rag_prompt, non_rag_prompt

def get_important_heads_from_text(model: TransformerLensTransparentLlm, text: str) -> List[Tuple[int, int]]:
    """
    Exact implementation of the reference logic to find important heads.
    """
    model._model.reset_hooks()
    model.run(text)
    
    tokens = model._model.to_tokens(text)
    tokens_size = tokens.shape[-1]
    
    # Generate all source->target pairs (causal masking)
    pos_idx_pairs = [(src, tgt) for src in range(tokens_size) for tgt in range(src, tokens_size)]
    
    important_heads = []
    
    for layer in range(N_LAYERS):
        for src_token_idx, target_token_idx in pos_idx_pairs:
            # Calculate contributions using LLM-TT
            head_contrib, c_resid_attn = contributions.get_attention_contributions(
                resid_pre=model.residual_in(layer)[0].unsqueeze(0),
                resid_mid=model.residual_after_attn(layer)[0].unsqueeze(0),
                decomposed_attn=model.decomposed_attn(0, layer).unsqueeze(0),
            )
            
            # Extract contribution vector for this specific token pair
            flat_contrib = head_contrib[0, target_token_idx, src_token_idx, :]
            
            # Apply threshold exactly as in reference
            important_head_idx = torch.nonzero(flat_contrib > CONTRIBUTION_THRESHOLD, as_tuple=False).squeeze(-1).tolist()
            
            if important_head_idx != []:
                # Ensure it's a list even if single element
                if isinstance(important_head_idx, int):
                    important_head_idx = [important_head_idx]
                    
                for head_idx in important_head_idx:
                    # Store as unique set to avoid duplicates per prompt, or keep duplicates if tracking frequency
                    # Reference code appends, so we append.
                    important_heads.append((layer, head_idx))
                    
    return important_heads


In [7]:
ds = load_dataset(DATASET_ID, split="validation")

df = pd.DataFrame({
    'question': ds['question'],
    'answers': ds['answers'], # List of valid answers
    'context': ds['text']     # The Gold Paragraph
})

In [9]:
# tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
# model = TransformerLensTransparentLlm(MODEL_NAME)
# model._model.to(device);
# model._model.to(torch.bfloat16).to(device);

# tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# bnb_config = BitsAndBytesConfig(
#         load_in_4bit=True,
#         bnb_4bit_use_double_quant=True,
#         bnb_4bit_quant_type="nf4",
#         bnb_4bit_compute_dtype=torch.bfloat16
#     )

# hf_model = AutoModelForCausalLM.from_pretrained(
#     MODEL_NAME,
#     quantization_config=bnb_config,
#     torch_dtype=torch.bfloat16,
#     device_map="auto"
# )
# tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# model = TransformerLensTransparentLlm(
#     model_name=MODEL_NAME,
#     hf_model=hf_model,
#     tokenizer=tokenizer,
#     device="gpu",
#     dtype=torch.bfloat16
# )

hf_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.bfloat16,
        device_map="auto" 
    )
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Initialize wrapper with the pre-loaded optimized model
model = TransformerLensTransparentLlm(
    model_name=MODEL_NAME,
    hf_model=hf_model,
    tokenizer=tokenizer,
    device="gpu" if torch.cuda.is_available() else "cpu",
    dtype=torch.bfloat16
)

In [10]:
data_sample = df.sample(n=100, random_state=42).to_dict(orient='records')

# Counters for different analysis modes
rag_unique_heads_counter = collections.defaultdict(int)
rag_total_counter = collections.defaultdict(int)
non_rag_total_counter = collections.defaultdict(int)

print(f"3. Running Analysis on {len(data_sample)} samples...")

for i, sample in enumerate(tqdm(data_sample)):
    question = sample['question']
    context = sample['context'] 
    
    # try:
    rag_prompt, non_rag_prompt = construct_prompts(question, context)
    
    # --- Get Important Heads for RAG ---
    rag_heads = get_important_heads_from_text(model, rag_prompt)
    rag_heads_set = set(rag_heads)
    
    # --- Get Important Heads for Non-RAG ---
    non_rag_heads = get_important_heads_from_text(model, non_rag_prompt)
    non_rag_heads_set = set(non_rag_heads)
    
    # --- 1. Total RAG Analysis ---
    for (layer, head) in rag_heads_set:
        rag_total_counter[(layer, head)] += 1

    # --- 2. Total Non-RAG Analysis ---
    for (layer, head) in non_rag_heads_set:
        non_rag_total_counter[(layer, head)] += 1

    # --- 3. Differential Analysis (Unique to RAG) ---
    # Find heads that are "Important" in RAG but NOT in Non-RAG
    unique_rag_heads = rag_heads_set - non_rag_heads_set
    for (layer, head) in unique_rag_heads:
        rag_unique_heads_counter[(layer, head)] += 1
                
    # except Exception as e:
    #     print(f"Error processing sample {i}: {e}. Skipping.")
    #     continue

# 4. Final Reporting helper
def save_and_print_top_heads(counter, title):
    if len(counter) > 0:
        sorted_heads = sorted(counter.items(), key=lambda item: item[1], reverse=True)
        
        print("\n" + "="*80)
        print(title)
        print("="*80)
        print(f"{'Layer':<5} {'Head':<5} {'Frequency':<12}")
        print("-" * 35)
        for (layer, head), count in sorted_heads[:15]:
            print(f"{layer:<5} {head:<5} {count:<12}")
        
        return {f"L{l}_H{h}": count for (l, h), count in sorted_heads}
    return {}

# Collect and Print Results
all_results = {}

print("\n--- RESULTS SUMMARY ---")
all_results["rag_unique"] = save_and_print_top_heads(
    rag_unique_heads_counter, 
    "TOP HEADS UNIQUE TO RAG (Differential - Context Processing)"
)
all_results["rag_total"] = save_and_print_top_heads(
    rag_total_counter, 
    "TOP HEADS IN RAG (Total Activity)"
)
all_results["non_rag_total"] = save_and_print_top_heads(
    non_rag_total_counter, 
    "TOP HEADS IN NON-RAG (Parametric/Closed-Book)"
)

3. Running Analysis on 100 samples...


  0%|                                                                                                                                                                                                                                                                                                                                                           | 0/100 [00:00<?, ?it/s]

Loaded pretrained model meta-llama/Llama-3.2-1B-Instruct into HookedTransformer
Loaded pretrained model meta-llama/Llama-3.2-1B-Instruct into HookedTransformer
Loaded pretrained model meta-llama/Llama-3.2-1B-Instruct into HookedTransformer
Loaded pretrained model meta-llama/Llama-3.2-1B-Instruct into HookedTransformer
Loaded pretrained model meta-llama/Llama-3.2-1B-Instruct into HookedTransformer
Loaded pretrained model meta-llama/Llama-3.2-1B-Instruct into HookedTransformer


  0%|                                                                                                                                                                                                                                                                                                                                                           | 0/100 [00:21<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 10.48 GiB. GPU 0 has a total capacity of 39.49 GiB of which 9.09 GiB is free. Including non-PyTorch memory, this process has 30.39 GiB memory in use. Of the allocated memory 29.86 GiB is allocated by PyTorch, and 41.55 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:

aggregate_diff_flow = collections.defaultdict(float)

print(f"3. Running Differential Attribution on {len(data_sample)} samples...")
for i, sample in enumerate(tqdm(data_sample)):
    question = sample['question']
    context = sample['text'] 
    target_answer = sample['answers'][0] 
    
    try:
        target_token_id = get_target_token_id(tokenizer, target_answer)
        rag_prompt, non_rag_prompt = construct_prompts(question, context)
        
        # --- RAG Attribution (Context Present) ---
        rag_relevance = calculate_head_contributions(model, rag_prompt, target_token_id)
        
        # --- Non-RAG Attribution (Parametric only) ---
        non_rag_relevance = calculate_head_contributions(model, non_rag_prompt, target_token_id)
        
        # --- Differential Flow ---
        for layer in range(N_LAYERS):
            for head in range(N_HEADS):
                head_key = (layer, head)
                diff = rag_relevance.get(head_key, 0.0) - non_rag_relevance.get(head_key, 0.0)
                aggregate_diff_flow[head_key] += diff
                
    except Exception as e:
        # Catch errors due to prompt length, tokenization, or internal LLM-TT issues
        print(f"Error processing sample {i}: {e}. Skipping.")
        continue

# 4. Final Aggregation and Reporting
num_processed = len(data_sample)
if num_processed > 0:
    avg_diff_flow = {k: v / num_processed for k, v in aggregate_diff_flow.items()}
    sorted_diff = sorted(avg_diff_flow.items(), key=lambda item: abs(item[1]), reverse=True)
    
    # Save results (optional, but good practice)
    results_to_save = {f"L{l}_H{h}": score for (l, h), score in avg_diff_flow.items()}
    with open(OUTPUT_FILE, 'w') as f:
        json.dump(results_to_save, f, indent=4)
    
    print("\n" + "="*80)
    print(f"DIFFERENTIAL IFR ANALYSIS COMPLETE. Results saved to {OUTPUT_FILE}")
    print("TOP 10 ATTENTION HEADS BY DIFFERENTIAL FLOW (RAG - Non-RAG)")
    print("="*80)
    
    print(f"{'Layer':<5} {'Head':<5} {'Diff Score':<12} {'Inferred Role'}")
    print("-" * 35)
    for (layer, head), diff in sorted_diff[:10]:
        role = "Context Integration (RAG Focus)" if diff > 0 else "Parametric Recall (Closed Focus)"
        print(f"{layer:<5} {head:<5} {diff:.6f} {role}")
        
    print("\n")

print("\n--- Next Critical Step: Hallucination Case Study ---")
print("This current run identifies general RAG-specific heads.")
print("To find HALLUCINATION HEADS, you must manually curate the NQ samples where:")
print("   1. The RAG context is correct (Gold Document).")
print("   2. The LLM's RAG output is factually incorrect (Hallucination).")
print("Then, compare the flow between: RAG (Hallucination Case) vs. RAG (Correct Case).")
print("This will reveal the circuit that fires when context is present but ignored.")


if __name__ == "__main__":
# Ensure torch runs on GPU if available
if not torch.cuda.is_available():
    print("Warning: CUDA not available. Running on CPU may be extremely slow.")
    
# Start the experiment
analyze_nq_samples()